In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

In [2]:
data = pd.read_csv("patient_data_processed.csv")

X = data.drop(columns=['Patient', 'Stages', 'Systolic', 'Diastolic'])
y = data['Stages']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=9))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ),
    
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel='linear'))
    ]),
    
    "RBF SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel='rbf'))
    ])
}

In [5]:
results = []

for name, model in models.items():
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    cv_scores = cross_val_score(model, X, y, cv=5)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    results.append([name, acc, prec, rec, f1, cv_mean, cv_std])

comparison_df = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "CV Mean Accuracy",
    "CV Std Dev"
])

comparison_df.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy,Precision,Recall,F1-Score,CV Mean Accuracy,CV Std Dev
4,RBF SVM,0.791781,0.845250,0.791781,0.793472,0.711781,0.070925
0,Logistic Regression,0.786301,0.845468,0.786301,0.784219,0.712877,0.082035
3,Linear SVM,0.786301,0.845468,0.786301,0.784219,0.722740,0.118023
1,KNN,0.767123,0.790141,0.767123,0.772776,0.763836,0.107618
2,Random Forest,0.767123,0.794125,0.767123,0.771827,0.714521,0.092559


In [6]:
best_model_name = comparison_df.sort_values(by="Accuracy", ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)
y_pred_best = best_model.predict(X_test)

print("Best Model:", best_model_name)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_best))

Best Model: RBF SVM

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.63      0.71        67
           1       0.83      0.93      0.88       130
           2       1.00      0.65      0.79       120
           3       0.53      1.00      0.70        48

    accuracy                           0.79       365
   macro avg       0.80      0.80      0.77       365
weighted avg       0.85      0.79      0.79       365


Confusion Matrix:

[[ 42  25   0   0]
 [  9 121   0   0]
 [  0   0  78  42]
 [  0   0   0  48]]
